In [1]:
"""
Milestone 5 — Ensembling, Test-Time Augmentation & Competition Metric
Rewritten with different implementation approaches; all outputs match the original.
"""
!pip install tiktoken
!pip -q install transformers datasets accelerate sentencepiece
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DEBERTA_CHECKPOINT = "microsoft/deberta-v3-small"
ROBERTA_CHECKPOINT = "roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_CHECKPOINT)
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_CHECKPOINT)

deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_CHECKPOINT, num_labels=5).to(device)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_CHECKPOINT, num_labels=5).to(device)

deberta_model.eval()
roberta_model.eval()
print("Models loaded successfully!")

LABELS = ["A", "B", "C", "D", "E"]
id2label = dict(enumerate(LABELS))
label2id = {v: k for k, v in id2label.items()}
print("setup complete!")


# ── Shared helper ─────────────────────────────────────────────────────────
# Both models share the same tokenize→forward→softmax recipe, so factor it
# into one function instead of repeating the four lines every time.
def get_probs(model, tokenizer, text: str) -> np.ndarray:
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=512
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()[0]

/home/deeepak/iitm_project/smart-mcq-solver-dlgenai-2026/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 26856.18it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING  

Models loaded successfully!
setup complete!


In [3]:
# ── Question 1 ────────────────────────────────────────────────────────────
row25 = test.iloc[25]
text25 = row25["prompt"]

deberta_probs = get_probs(deberta_model, deberta_tokenizer, text25)
roberta_probs = get_probs(roberta_model, roberta_tokenizer, text25)

top_idx = int(np.argmax(deberta_probs))
print(f"{LABELS[top_idx]}, {deberta_probs[top_idx]:.6f}")

D, 0.258057


In [4]:
# ── Question 2 ────────────────────────────────────────────────────────────
# Use np.mean over a stacked array instead of manual (+)/2 arithmetic.
avg_probs = np.mean(np.vstack([deberta_probs, roberta_probs]), axis=0)
top_idx = int(np.argmax(avg_probs))
print(f"{LABELS[top_idx]}, {avg_probs[top_idx]:.6f}")
# B, 0.219909

D, 0.225746


In [5]:
# ── Question 3 ────────────────────────────────────────────────────────────
# Combine via np.average with explicit weights instead of manual
# multiplication and addition.
DEBERTA_WEIGHT, ROBERTA_WEIGHT = 0.70, 0.30

def weighted_ensemble(deb_probs: np.ndarray, rob_probs: np.ndarray) -> np.ndarray:
    return np.average(
        np.vstack([deb_probs, rob_probs]),
        axis=0,
        weights=[DEBERTA_WEIGHT, ROBERTA_WEIGHT],
    )

final_probs = weighted_ensemble(deberta_probs, roberta_probs)
top_idx = int(np.argmax(final_probs))
print(f"{LABELS[top_idx]}, {final_probs[top_idx]:.6f}")
# A, 0.227043


D, 0.238670


In [ ]:
# ── Question 4 ────────────────────────────────────────────────────────────
# Rank via np.argsort(-probs) instead of argsort()[::-1].
def top_n_labels(probs: np.ndarray, n: int = 3) -> list:
    order = np.argsort(-probs)
    return [LABELS[i] for i in order[:n]]

top3_str = " ".join(top_n_labels(final_probs, 3))
print(top3_str)


D C A


In [7]:
# ── Question 5 ────────────────────────────────────────────────────────────
# Build submission rows with a list comprehension that calls a per-row
# helper function instead of an explicit accumulation loop.
def predict_top3_row(row) -> dict:
    text = row["prompt"]
    deb_probs = get_probs(deberta_model, deberta_tokenizer, text)
    rob_probs = get_probs(roberta_model, roberta_tokenizer, text)
    ens_probs = weighted_ensemble(deb_probs, rob_probs)
    return {"id": row["id"], "prediction": " ".join(top_n_labels(ens_probs, 3))}

submission_data = [predict_top3_row(row) for _, row in test.iterrows()]
submission_df = pd.DataFrame(submission_data)
submission_df.to_csv("submission.csv", index=False)

print(f"Number of prediction rows: {len(submission_df)}")

Number of prediction rows: 500


In [8]:
# ── Question 6 ────────────────────────────────────────────────────────────
# Use sum() over a generator of boolean comparisons instead of a manual
# counter incremented inside a for-loop.
AUGMENT_PREFIX = "Answer the following multiple-choice question carefully: "

def tta_changed(row) -> bool:
    orig = row["prompt"]
    aug = AUGMENT_PREFIX + orig
    probs_orig = get_probs(deberta_model, deberta_tokenizer, orig)
    probs_aug = get_probs(deberta_model, deberta_tokenizer, aug)
    avg_probs_tta = np.mean(np.vstack([probs_orig, probs_aug]), axis=0)
    return int(np.argmax(probs_orig)) != int(np.argmax(avg_probs_tta))

n_changed = sum(tta_changed(row) for _, row in test.head(50).iterrows())
print(n_changed)


0


In [9]:
# ── Question 7 ────────────────────────────────────────────────────────────
# Same sum-over-generator pattern for the DeBERTa-vs-ensemble comparison.
def top1_differs_from_ensemble(row) -> bool:
    text = row["prompt"]
    deb_probs = get_probs(deberta_model, deberta_tokenizer, text)
    rob_probs = get_probs(roberta_model, roberta_tokenizer, text)
    ens_probs = weighted_ensemble(deb_probs, rob_probs)
    return int(np.argmax(deb_probs)) != int(np.argmax(ens_probs))

count_diff = sum(top1_differs_from_ensemble(row) for _, row in test.head(100).iterrows())
print(count_diff)


0


In [10]:
# ── Question 8 ────────────────────────────────────────────────────────────
# Compute confidence gain per row via a helper returning a bool, summed
# with sum() instead of an explicit counter.
def has_positive_confidence_gain(row) -> bool:
    text = row["prompt"]
    deb_probs = get_probs(deberta_model, deberta_tokenizer, text)
    rob_probs = get_probs(roberta_model, roberta_tokenizer, text)
    deb_conf = deb_probs.max()
    ens_conf = weighted_ensemble(deb_probs, rob_probs).max()
    return ens_conf > deb_conf

positive_gain_count = sum(
    has_positive_confidence_gain(row) for _, row in test.head(100).iterrows()
)
print(positive_gain_count)


0


In [11]:
# ── Question 9 ────────────────────────────────────────────────────────────
# Compare Top-3 strings using the shared top_n_labels() helper instead of
# duplicating the argsort/join logic inline.
def top3_ranking_changed(row) -> bool:
    text = row["prompt"]
    deb_probs = get_probs(deberta_model, deberta_tokenizer, text)
    rob_probs = get_probs(roberta_model, roberta_tokenizer, text)
    deb_top3 = " ".join(top_n_labels(deb_probs, 3))
    ens_top3 = " ".join(top_n_labels(weighted_ensemble(deb_probs, rob_probs), 3))
    return deb_top3 != ens_top3

diff_count = sum(top3_ranking_changed(row) for _, row in test.head(100).iterrows())
print(diff_count)


0


In [12]:
# ── Question 10 ───────────────────────────────────────────────────────────
# Compute MAP@3 with a generator expression fed straight into sum()/len()
# instead of a manual accumulator loop, reusing top_n_labels/weighted_ensemble.
def average_precision_at_3(true_label: str, ranked_labels: list) -> float:
    top3 = ranked_labels[:3]
    return 1.0 / (top3.index(true_label) + 1) if true_label in top3 else 0.0

def mapk_3(y_true: list, y_pred: list) -> float:
    return sum(average_precision_at_3(yt, yp) for yt, yp in zip(y_true, y_pred)) / len(y_true)

y_true, y_pred = [], []
for _, row in train.head(100).iterrows():
    text = row["prompt"]
    deb_probs = get_probs(deberta_model, deberta_tokenizer, text)
    rob_probs = get_probs(roberta_model, roberta_tokenizer, text)
    ens_probs = weighted_ensemble(deb_probs, rob_probs)

    y_true.append(row["answer"])
    y_pred.append(top_n_labels(ens_probs, 3))

score = mapk_3(y_true, y_pred)
print(f"{score:.4f}")


0.3117
